# CS5224 Data Curation — Stage 2D

## Expand the validated identity-matching process to all eight categories and build a  shortlist

Stage 2C validated the automated/semi-automated identity workflow on the three pilot categories.

Stage 2D now:

1. checks that Stage 2C passed the QA gate;
2. expands the same conservative candidate-generation and identity rules to **all eight agreed categories**;
3. preserves Stage 2C reviewed decisions as authoritative overrides for pilot candidate pairs;
4. constructs all-category provisional canonical products;
5. quarantines ambiguous/non-match/pack-discrepancy cases;
6. creates a  **1300-product provisional shortlist** across the eight categories;
7. creates a focused fulfilment-verification queue only for listings belonging to the shortlist.

### Important distinction

`offer_eligible` still represents the source-snapshot stock/quality gate from Stage 1E.

It does **not** prove direct supermarket fulfilment.

Therefore the Stage 2D shortlist uses `offer_eligible` only as a **preliminary prioritisation signal**. The final requirement of eligible offers from 2+ retailers can only be assessed after direct fulfilment is verified.

### Default shortlist configuration

The implementation uses:

- `TARGET_SHORTLIST_SIZE = 1300`
- `MIN_PRODUCTS_PER_CATEGORY = 50`
- `MAX_PRODUCTS_PER_CATEGORY = 450`

These are implementation choices for a balanced provisional seed within the required final range of 200–300 products. They can be adjusted without changing the matching logic.


## 1. Imports, project paths, and target categories

In [1]:
from pathlib import Path
from difflib import SequenceMatcher
from urllib.parse import urlparse
import hashlib
import re

import numpy as np
import pandas as pd

PROJECT_ROOT = Path.cwd().parent
STAGING_DATA_DIR = PROJECT_ROOT / "data" / "staging"
CURATED_DATA_DIR = PROJECT_ROOT / "data" / "curated"

STAGE2A_MAPPED_FILE = (
    STAGING_DATA_DIR
    / "all_retailers_stage2a_category_mapped.csv"
)

STAGE2C_SUMMARY_FILE = (
    CURATED_DATA_DIR
    / "stage2c_automation_summary.csv"
)

STAGE2C_FINAL_PAIRS_FILE = (
    CURATED_DATA_DIR
    / "stage2c_final_identity_pairs.csv"
)

STAGE2C_FULFILMENT_FILE = (
    CURATED_DATA_DIR
    / "stage2c_fulfilment_review_working.csv"
)

TARGET_CATEGORIES = [
    "Rice & Noodles",
    "Drinks",
    "Dairy",
    "Breakfast Foods",
    "Canned & Packaged Food",
    "Snacks",
    "Household Cleaning",
    "Toiletries & Personal Care",
]

for file_path in [
    STAGE2A_MAPPED_FILE,
    STAGE2C_SUMMARY_FILE,
    STAGE2C_FINAL_PAIRS_FILE,
]:
    print(
        f"{file_path.name:52s} -> "
        f"{file_path.exists()}"
    )
    assert file_path.exists()

print(
    "\nPASS: Required Stage 2A and Stage 2C files found."
)


all_retailers_stage2a_category_mapped.csv            -> True
stage2c_automation_summary.csv                       -> True
stage2c_final_identity_pairs.csv                     -> True

PASS: Required Stage 2A and Stage 2C files found.


## 2. Validate the Stage 2C QA gate before expansion

In [2]:
stage2c_summary = pd.read_csv(
    STAGE2C_SUMMARY_FILE,
    low_memory=False,
)

display(stage2c_summary)

def get_summary_value(
    metric_name,
    preferred_column,
):
    rows = stage2c_summary.loc[
        stage2c_summary[
            "metric_name"
        ]
        == metric_name
    ]

    assert len(rows) == 1, (
        f"Expected exactly one summary row for "
        f"{metric_name}, found {len(rows)}."
    )

    value = rows.iloc[0][
        preferred_column
    ]

    return value


qa_family_agreement_rate = float(
    get_summary_value(
        "QA_FAMILY_AGREEMENT_RATE",
        "value",
    )
)

qa_unresolved_conflicts = int(
    float(
        get_summary_value(
            "QA_UNRESOLVED_MATERIAL_CONFLICTS",
            "count",
        )
    )
)

MIN_QA_FAMILY_AGREEMENT_RATE = 0.95

print(
    "Stage 2C QA family agreement rate:",
    f"{qa_family_agreement_rate:.2%}"
)

print(
    "Stage 2C unresolved material conflicts:",
    qa_unresolved_conflicts
)

assert (
    qa_family_agreement_rate
    >= MIN_QA_FAMILY_AGREEMENT_RATE
), (
    "Stage 2C QA agreement is below the expansion gate."
)

assert qa_unresolved_conflicts == 0, (
    "Resolve all material Stage 2C QA conflicts "
    "before Stage 2D expansion."
)

print(
    "\nPASS: Stage 2C QA gate passed."
)


,metric_type,metric_name,count,value
0,AUTO_IDENTITY_DECISION,NEEDS_MANUAL_REVIEW,7026.0,NaN
1,AUTO_IDENTITY_DECISION,AUTO_NOT_SAME_PRODUCT,1602.0,NaN
2,AUTO_IDENTITY_DECISION,AUTO_VARIANT_MISMATCH,1112.0,NaN
3,AUTO_IDENTITY_DECISION,AUTO_MATCH_CONFIRMED,910.0,NaN
4,FINAL_IDENTITY_DECISION,UNREVIEWED,7026.0,NaN
5,FINAL_IDENTITY_DECISION,NOT_SAME_PRODUCT,1572.0,NaN
6,FINAL_IDENTITY_DECISION,VARIANT_MISMATCH,1140.0,NaN
7,FINAL_IDENTITY_DECISION,MATCH_CONFIRMED,912.0,NaN
8,WORKLOAD,TOTAL_CANDIDATE_PAIRS,10650.0,NaN
9,WORKLOAD,MANUAL_REVIEW_REQUIRED,7026.0,NaN


Stage 2C QA family agreement rate: 97.78%
Stage 2C unresolved material conflicts: 0

PASS: Stage 2C QA gate passed.


## 3. Load the full Stage 2A mapped dataset and Stage 2C reviewed decisions

In [3]:
stage2a = pd.read_csv(
    STAGE2A_MAPPED_FILE,
    low_memory=False,
)

stage2c_final_pairs = pd.read_csv(
    STAGE2C_FINAL_PAIRS_FILE,
    low_memory=False,
)

print(
    "Stage 2A mapped records:",
    f"{len(stage2a):,}"
)

print(
    "Stage 2C final candidate pairs:",
    f"{len(stage2c_final_pairs):,}"
)

display(stage2a.head(3))


Stage 2A mapped records: 177,638
Stage 2C final candidate pairs: 10,650


,retailer,source_file,source_sheet,source_row_number,source_record_id,source_category_raw,source_category_normalized,product_name_raw,product_name_normalized,brand_raw,...,eligibility_status,source_category_match_count,product_name_match_count,target_category_suggested,target_category_final,category_mapping_status,category_mapping_confidence,category_mapping_method,source_category_candidates,product_name_candidates
0,redmart,redmart_all_products_final.xlsx,Sheet1,2,redmart_00000002,shop-chilled-desserts,shop-chilled-desserts,Sojasun Natural Soya Yoghurt,sojasun natural soya yoghurt,Sojasun,...,ELIGIBLE_PRELIMINARY,1,1,Snacks,Snacks,AUTO_SOURCE_CATEGORY,HIGH,SOURCE_CATEGORY_RULE,Snacks,Dairy
1,redmart,redmart_all_products_final.xlsx,Sheet1,3,redmart_00000003,shop-chilled-desserts,shop-chilled-desserts,Monoprix Chocolate Custard Dessert Cream 4x125G,monoprix chocolate custard dessert cream 4x125g,Monoprix,...,ELIGIBLE_PRELIMINARY,1,1,Snacks,Snacks,AUTO_SOURCE_CATEGORY,HIGH,SOURCE_CATEGORY_RULE,Snacks,Snacks
2,redmart,redmart_all_products_final.xlsx,Sheet1,4,redmart_00000004,shop-chilled-desserts,shop-chilled-desserts,Top Gourmet Cheng Tng,top gourmet cheng tng,Top Gourmet,...,ELIGIBLE_PRELIMINARY,1,0,Snacks,Snacks,AUTO_SOURCE_CATEGORY,HIGH,SOURCE_CATEGORY_RULE,Snacks,NONE


## 4. Restore important nullable data types

In [4]:
def to_nullable_boolean(series):
    normalized = (
        series
        .astype("string")
        .str.strip()
        .str.casefold()
    )

    mapping = {
        "true": True,
        "false": False,
        "yes": True,
        "no": False,
        "1": True,
        "0": False,
    }

    return (
        normalized
        .map(mapping)
        .astype("boolean")
    )


for column in [
    "in_stock",
    "currently_available",
    "has_offer",
    "needs_manual_review",
    "basic_eligibility",
    "catalog_eligible",
    "offer_eligible",
]:
    if column in stage2a.columns:
        stage2a[column] = (
            to_nullable_boolean(
                stage2a[column]
            )
        )


if "pack_count" in stage2a.columns:
    stage2a["pack_count"] = (
        pd.to_numeric(
            stage2a["pack_count"],
            errors="coerce",
        )
        .astype("Int64")
    )


for column in [
    "unit_size",
    "total_quantity",
    "total_quantity_base",
    "product_price",
    "original_price",
]:
    if column in stage2a.columns:
        stage2a[column] = (
            pd.to_numeric(
                stage2a[column],
                errors="coerce",
            )
            .astype("Float64")
        )


## 5. Build the all-category expansion pool

Only records that were:

- automatically mapped from a unique high-confidence source-category rule; and
- catalog-eligible in Stage 1E

are admitted automatically.

Product-name category suggestions and category conflicts remain outside the expansion pool.


In [5]:
REQUIRED_STAGE2A_COLUMNS = [
    "target_category_final",
    "category_mapping_status",
    "retailer",
    "source_record_id",
    "product_name_raw",
    "product_name_normalized",
    "brand_normalized",
    "specification_raw",
    "pack_count",
    "unit_size",
    "unit",
    "total_quantity",
    "total_quantity_base",
    "base_unit",
    "product_price",
    "stock_status",
    "currently_available",
    "catalog_eligible",
    "offer_eligible",
    "product_url_raw",
]

missing_columns = [
    column
    for column in REQUIRED_STAGE2A_COLUMNS
    if column not in stage2a.columns
]

assert not missing_columns, (
    f"Missing required Stage 2A columns: "
    f"{missing_columns}"
)

expansion_pool = (
    stage2a.loc[
        stage2a[
            "target_category_final"
        ].isin(
            TARGET_CATEGORIES
        )
        &
        (
            stage2a[
                "category_mapping_status"
            ]
            == "AUTO_SOURCE_CATEGORY"
        )
        &
        stage2a[
            "catalog_eligible"
        ].fillna(False)
    ]
    .copy()
)

assert (
    expansion_pool[
        "source_record_id"
    ]
    .duplicated()
    .sum()
    == 0
)

assert (
    expansion_pool[
        "brand_normalized"
    ]
    .notna()
    .all()
)

assert (
    expansion_pool[
        "target_category_final"
    ]
    .notna()
    .all()
)

categories_present = set(
    expansion_pool[
        "target_category_final"
    ]
    .unique()
)

missing_target_categories = (
    set(TARGET_CATEGORIES)
    - categories_present
)

print(
    "All-category expansion pool:",
    f"{len(expansion_pool):,}"
)

print(
    "Categories present:",
    len(categories_present)
)

print(
    "Missing target categories:",
    sorted(
        missing_target_categories
    )
)

display(
    pd.crosstab(
        expansion_pool[
            "target_category_final"
        ],
        expansion_pool[
            "retailer"
        ],
        margins=True,
    )
)

assert not missing_target_categories, (
    "All eight target categories must be represented "
    "before expansion."
)

print(
    "\nPASS: All eight categories are represented."
)


All-category expansion pool: 49,690
Categories present: 8
Missing target categories: []


retailer,coldstorage,fairprice,redmart,All
target_category_final,,,,
Breakfast Foods,248,333,951,1532
Canned & Packaged Food,628,1357,5010,6995
Dairy,446,786,2825,4057
Drinks,1040,2145,6219,9404
Household Cleaning,38,1366,2234,3638
Rice & Noodles,357,1877,2340,4574
Snacks,1262,3547,10387,15196
Toiletries & Personal Care,284,2932,1078,4294
All,4303,14343,31044,49690



PASS: All eight categories are represented.


## 6. Compute strict and relaxed pack keys

In [6]:
expansion = expansion_pool.copy()

expansion[
    "per_unit_quantity_base"
] = (
    expansion[
        "total_quantity_base"
    ]
    /
    expansion[
        "pack_count"
    ].astype("Float64")
)

expansion[
    "per_unit_quantity_base_key"
] = (
    expansion[
        "per_unit_quantity_base"
    ]
    .round(6)
)

expansion[
    "total_quantity_base_key"
] = (
    expansion[
        "total_quantity_base"
    ]
    .round(6)
)

assert (
    expansion[
        "per_unit_quantity_base"
    ] > 0
).all()


def safe_key_value(value):
    if pd.isna(value):
        return "<NA>"

    if isinstance(
        value,
        (float, np.floating),
    ):
        return f"{float(value):.6f}"

    return str(value)


expansion[
    "strict_pack_key"
] = (
    expansion.apply(
        lambda row: "|".join(
            [
                safe_key_value(
                    row[
                        "base_unit"
                    ]
                ),
                safe_key_value(
                    row[
                        "pack_count"
                    ]
                ),
                safe_key_value(
                    row[
                        "per_unit_quantity_base_key"
                    ]
                ),
                safe_key_value(
                    row[
                        "total_quantity_base_key"
                    ]
                ),
            ]
        ),
        axis=1,
    )
    .astype("string")
)

expansion[
    "total_pack_key"
] = (
    expansion.apply(
        lambda row: "|".join(
            [
                safe_key_value(
                    row[
                        "base_unit"
                    ]
                ),
                safe_key_value(
                    row[
                        "total_quantity_base_key"
                    ]
                ),
            ]
        ),
        axis=1,
    )
    .astype("string")
)

print(
    "PASS: Strict and relaxed pack keys created."
)


PASS: Strict and relaxed pack keys created.


## 7. Build product-name comparison text using the validated Stage 2B representation

In [7]:
MATCH_STOPWORDS = {
    "the",
    "and",
    "with",
    "for",
    "of",
    "by",
    "a",
    "an",
    "pack",
    "packs",
    "pkt",
    "ctn",
    "carton",
    "pcs",
    "pc",
    "piece",
    "pieces",
    "per",
    "each",
    "x",
    "g",
    "kg",
    "mg",
    "ml",
    "l",
}


def normalize_match_text(
    product_name,
    brand,
):
    if pd.isna(
        product_name
    ):
        return ""

    text = (
        str(
            product_name
        )
        .casefold()
    )

    if (
        not pd.isna(
            brand
        )
        and str(
            brand
        ).strip()
    ):
        brand_text = (
            str(
                brand
            )
            .casefold()
            .strip()
        )

        text = text.replace(
            brand_text,
            " ",
        )

    text = re.sub(
        r"\b\d+(?:\.\d+)?\s*"
        r"(?:x|×)\s*"
        r"\d+(?:\.\d+)?\s*"
        r"(?:mg|g|kg|ml|l|pcs?|pieces?)\b",
        " ",
        text,
        flags=re.IGNORECASE,
    )

    text = re.sub(
        r"\b\d+(?:\.\d+)?\s*"
        r"(?:mg|g|kg|ml|l|pcs?|pieces?)\b",
        " ",
        text,
        flags=re.IGNORECASE,
    )

    text = re.sub(
        r"[^a-z0-9]+",
        " ",
        text,
    )

    tokens = [
        token
        for token in text.split()
        if token
        not in MATCH_STOPWORDS
    ]

    return " ".join(
        tokens
    )


expansion[
    "name_match_text"
] = [
    normalize_match_text(
        product_name,
        brand,
    )
    for product_name, brand
    in zip(
        expansion[
            "product_name_normalized"
        ],
        expansion[
            "brand_normalized"
        ],
    )
]

expansion[
    "name_match_tokens"
] = (
    expansion[
        "name_match_text"
    ]
    .apply(
        lambda text: tuple(
            sorted(
                set(
                    str(
                        text
                    ).split()
                )
            )
        )
    )
)

display(
    expansion[
        [
            "target_category_final",
            "product_name_raw",
            "brand_normalized",
            "name_match_text",
            "strict_pack_key",
        ]
    ].head(20)
)


,target_category_final,product_name_raw,brand_normalized,name_match_text,strict_pack_key
0,Snacks,Sojasun Natural Soya Yoghurt,sojasun,natural soya yoghurt,g|6|100.000000|600.000000
1,Snacks,Monoprix Chocolate Custard Dessert Cream 4x125G,monoprix,chocolate custard dessert cream,g|4|125.000000|500.000000
2,Snacks,Top Gourmet Cheng Tng,top gourmet,cheng tng,g|1|400.000000|400.000000
3,Snacks,La Laitiere Vanilla Rice Pudding Riz Au Lait 4...,la laitiere,vanilla rice pudding riz au lait,g|4|115.000000|460.000000
4,Snacks,Monoprix Flan Caramel Vanilla Pudding 4X100G,monoprix,flan caramel vanilla pudding,g|4|100.000000|400.000000
5,Snacks,Fortune Sweetened Chin Chow 1KG,fortune,sweetened chin chow,g|1|1000.000000|1000.000000
6,Snacks,RedMart Trio Swiss Roll (Singapore Special),redmart,trio swiss roll singapore special,g|1|430.000000|430.000000
7,Snacks,Arla Protein Chocolate Pudding 200G - Lactose ...,arla,protein chocolate pudding lactose free,g|1|200.000000|200.000000
8,Snacks,Fortune Soya Beancurd Almond 360G,fortune,soya beancurd almond,g|1|360.000000|360.000000
9,Snacks,Arla Protein Hazelnut Pudding 200G - Lactose Free,arla,protein hazelnut pudding lactose free,g|1|200.000000|200.000000


## 8. Define name similarity and retailer pairs

In [8]:
def token_jaccard(
    left_text,
    right_text,
):
    left = set(
        str(
            left_text
        ).split()
    )

    right = set(
        str(
            right_text
        ).split()
    )

    if (
        not left
        and not right
    ):
        return 1.0

    if (
        not left
        or not right
    ):
        return 0.0

    return (
        len(
            left
            & right
        )
        /
        len(
            left
            | right
        )
    )


def sequence_similarity(
    left_text,
    right_text,
):
    return SequenceMatcher(
        None,
        str(
            left_text
        ),
        str(
            right_text
        ),
    ).ratio()


def score_name_pair(
    left_text,
    right_text,
):
    jaccard = token_jaccard(
        left_text,
        right_text,
    )

    sequence = sequence_similarity(
        left_text,
        right_text,
    )

    exact_core = (
        str(
            left_text
        )
        == str(
            right_text
        )
        and str(
            left_text
        )
        != ""
    )

    combined = (
        0.60
        * jaccard
        + 0.40
        * sequence
    )

    return (
        jaccard,
        sequence,
        combined,
        exact_core,
    )


RETAILER_PAIRS = [
    (
        "redmart",
        "fairprice",
    ),
    (
        "redmart",
        "coldstorage",
    ),
    (
        "fairprice",
        "coldstorage",
    ),
]

available_retailers = set(
    expansion[
        "retailer"
    ]
    .dropna()
    .unique()
)

for (
    left_retailer,
    right_retailer,
) in RETAILER_PAIRS:
    assert (
        left_retailer
        in available_retailers
    )
    assert (
        right_retailer
        in available_retailers
    )

print(
    "PASS: Similarity functions and retailer pairs ready."
)


PASS: Similarity functions and retailer pairs ready.


## 9. Build strict multi-retailer blocks

In [9]:
STRICT_BLOCK_COLUMNS = [
    "target_category_final",
    "brand_normalized",
    "strict_pack_key",
]

strict_block_summary = (
    expansion
    .groupby(
        STRICT_BLOCK_COLUMNS,
        dropna=False,
    )
    .agg(
        total_records=(
            "source_record_id",
            "size",
        ),
        retailer_count=(
            "retailer",
            "nunique",
        ),
        redmart_records=(
            "retailer",
            lambda s: int(
                (
                    s
                    == "redmart"
                ).sum()
            ),
        ),
        fairprice_records=(
            "retailer",
            lambda s: int(
                (
                    s
                    == "fairprice"
                ).sum()
            ),
        ),
        coldstorage_records=(
            "retailer",
            lambda s: int(
                (
                    s
                    == "coldstorage"
                ).sum()
            ),
        ),
    )
    .reset_index()
)

multi_retailer_blocks = (
    strict_block_summary.loc[
        strict_block_summary[
            "retailer_count"
        ]
        >= 2
    ]
    .copy()
)

print(
    "Strict blocks:",
    f"{len(strict_block_summary):,}"
)

print(
    "Multi-retailer strict blocks:",
    f"{len(multi_retailer_blocks):,}"
)


Strict blocks: 20,780
Multi-retailer strict blocks: 2,445


## 10. Generate all-category strict candidate pairs

In [10]:
MAX_CARTESIAN_PAIRS_PER_BLOCK = 5_000
MAX_CANDIDATES_PER_SOURCE_PER_PAIR = 5
MIN_COMBINED_SIMILARITY_TO_KEEP = 0.25


def shared_meaningful_token(
    left_text,
    right_text,
):
    left = set(
        str(
            left_text
        ).split()
    )

    right = set(
        str(
            right_text
        ).split()
    )

    return bool(
        left
        & right
    )


def make_pair_record(
    left_row,
    right_row,
    left_retailer,
    right_retailer,
):
    (
        jaccard,
        sequence,
        combined,
        exact_core,
    ) = score_name_pair(
        left_row[
            "name_match_text"
        ],
        right_row[
            "name_match_text"
        ],
    )

    if exact_core:
        priority = (
            "VERY_HIGH"
        )

    elif (
        jaccard
        >= 0.80
        and sequence
        >= 0.80
    ):
        priority = "HIGH"

    elif combined >= 0.65:
        priority = "MEDIUM"

    else:
        priority = "LOW"

    left_id = str(
        left_row[
            "source_record_id"
        ]
    )

    right_id = str(
        right_row[
            "source_record_id"
        ]
    )

    pair_id = (
        f"{left_id}"
        f"__{right_id}"
    )

    return {
        "candidate_pair_id":
            pair_id,
        "retailer_pair":
            (
                f"{left_retailer}"
                f"__{right_retailer}"
            ),
        "target_category":
            left_row[
                "target_category_final"
            ],
        "brand_normalized":
            left_row[
                "brand_normalized"
            ],
        "strict_pack_key":
            left_row[
                "strict_pack_key"
            ],

        "left_retailer":
            left_retailer,
        "left_source_record_id":
            left_id,
        "left_product_name":
            left_row[
                "product_name_raw"
            ],
        "left_name_match_text":
            left_row[
                "name_match_text"
            ],
        "left_specification":
            left_row[
                "specification_raw"
            ],
        "left_product_price":
            left_row[
                "product_price"
            ],
        "left_stock_status":
            left_row[
                "stock_status"
            ],
        "left_offer_eligible":
            left_row[
                "offer_eligible"
            ],
        "left_product_url":
            left_row[
                "product_url_raw"
            ],

        "right_retailer":
            right_retailer,
        "right_source_record_id":
            right_id,
        "right_product_name":
            right_row[
                "product_name_raw"
            ],
        "right_name_match_text":
            right_row[
                "name_match_text"
            ],
        "right_specification":
            right_row[
                "specification_raw"
            ],
        "right_product_price":
            right_row[
                "product_price"
            ],
        "right_stock_status":
            right_row[
                "stock_status"
            ],
        "right_offer_eligible":
            right_row[
                "offer_eligible"
            ],
        "right_product_url":
            right_row[
                "product_url_raw"
            ],

        "token_jaccard":
            round(
                float(
                    jaccard
                ),
                6,
            ),
        "sequence_similarity":
            round(
                float(
                    sequence
                ),
                6,
            ),
        "combined_similarity":
            round(
                float(
                    combined
                ),
                6,
            ),
        "exact_name_core":
            bool(
                exact_core
            ),
        "candidate_priority":
            priority,
    }


candidate_records = []

grouped = expansion.groupby(
    STRICT_BLOCK_COLUMNS,
    dropna=False,
    sort=False,
)

for (
    block_key,
    block,
) in grouped:
    retailers_here = set(
        block[
            "retailer"
        ]
        .unique()
    )

    if len(
        retailers_here
    ) < 2:
        continue

    for (
        left_retailer,
        right_retailer,
    ) in RETAILER_PAIRS:
        if (
            left_retailer
            not in retailers_here
            or right_retailer
            not in retailers_here
        ):
            continue

        left_df = (
            block.loc[
                block[
                    "retailer"
                ]
                == left_retailer
            ]
        )

        right_df = (
            block.loc[
                block[
                    "retailer"
                ]
                == right_retailer
            ]
        )

        potential_pairs = (
            len(
                left_df
            )
            * len(
                right_df
            )
        )

        large_block = (
            potential_pairs
            > MAX_CARTESIAN_PAIRS_PER_BLOCK
        )

        for (
            _,
            left_row,
        ) in left_df.iterrows():
            local_candidates = []

            for (
                _,
                right_row,
            ) in right_df.iterrows():
                if (
                    large_block
                    and not shared_meaningful_token(
                        left_row[
                            "name_match_text"
                        ],
                        right_row[
                            "name_match_text"
                        ],
                    )
                ):
                    continue

                record = (
                    make_pair_record(
                        left_row,
                        right_row,
                        left_retailer,
                        right_retailer,
                    )
                )

                if (
                    record[
                        "combined_similarity"
                    ]
                    >= MIN_COMBINED_SIMILARITY_TO_KEEP
                ):
                    local_candidates.append(
                        record
                    )

            local_candidates = sorted(
                local_candidates,
                key=lambda row: row[
                    "combined_similarity"
                ],
                reverse=True,
            )[
                :MAX_CANDIDATES_PER_SOURCE_PER_PAIR
            ]

            candidate_records.extend(
                local_candidates
            )


strict_candidates = pd.DataFrame(
    candidate_records
)

if len(
    strict_candidates
) > 0:
    strict_candidates = (
        strict_candidates
        .drop_duplicates(
            subset=[
                "candidate_pair_id"
            ]
        )
        .reset_index(
            drop=True
        )
    )

print(
    "All-category strict candidate pairs:",
    f"{len(strict_candidates):,}"
)

display(
    strict_candidates.head(20)
)


All-category strict candidate pairs: 18,434


,candidate_pair_id,retailer_pair,target_category,brand_normalized,strict_pack_key,left_retailer,left_source_record_id,left_product_name,left_name_match_text,left_specification,...,right_specification,right_product_price,right_stock_status,right_offer_eligible,right_product_url,token_jaccard,sequence_similarity,combined_similarity,exact_name_core,candidate_priority
0,redmart_00000009__fairprice_00037226,redmart__fairprice,Snacks,arla,g|1|200.000000|200.000000,redmart,redmart_00000009,Arla Protein Chocolate Pudding 200G - Lactose ...,protein chocolate pudding lactose free,200 g,...,200g,3.80,IN_STOCK,True,https://www.fairprice.com.sg/product/arla-prot...,0.600000,0.539683,0.575873,False,LOW
1,redmart_00000009__fairprice_00037233,redmart__fairprice,Snacks,arla,g|1|200.000000|200.000000,redmart,redmart_00000009,Arla Protein Chocolate Pudding 200G - Lactose ...,protein chocolate pudding lactose free,200 g,...,200g,3.80,IN_STOCK,True,https://www.fairprice.com.sg/product/arla-prot...,0.285714,0.588235,0.406723,False,LOW
2,redmart_00000009__fairprice_00037232,redmart__fairprice,Snacks,arla,g|1|200.000000|200.000000,redmart,redmart_00000009,Arla Protein Chocolate Pudding 200G - Lactose ...,protein chocolate pudding lactose free,200 g,...,200g,3.80,IN_STOCK,True,https://www.fairprice.com.sg/product/arla-prot...,0.285714,0.558824,0.394958,False,LOW
3,redmart_00000011__fairprice_00037233,redmart__fairprice,Snacks,arla,g|1|200.000000|200.000000,redmart,redmart_00000011,Arla Protein Hazelnut Pudding 200G - Lactose Free,protein hazelnut pudding lactose free,200 g,...,200g,3.80,IN_STOCK,True,https://www.fairprice.com.sg/product/arla-prot...,0.500000,0.626866,0.550746,False,LOW
4,redmart_00000011__fairprice_00037226,redmart__fairprice,Snacks,arla,g|1|200.000000|200.000000,redmart,redmart_00000011,Arla Protein Hazelnut Pudding 200G - Lactose Free,protein hazelnut pudding lactose free,200 g,...,200g,3.80,IN_STOCK,True,https://www.fairprice.com.sg/product/arla-prot...,0.333333,0.645161,0.458065,False,LOW
5,redmart_00000011__fairprice_00037232,redmart__fairprice,Snacks,arla,g|1|200.000000|200.000000,redmart,redmart_00000011,Arla Protein Hazelnut Pudding 200G - Lactose Free,protein hazelnut pudding lactose free,200 g,...,200g,3.80,IN_STOCK,True,https://www.fairprice.com.sg/product/arla-prot...,0.285714,0.567164,0.398294,False,LOW
6,redmart_00000036__fairprice_00037232,redmart__fairprice,Snacks,arla,g|1|200.000000|200.000000,redmart,redmart_00000036,Arla Protein Salted Caramel Pudding 200G - Lac...,protein salted caramel pudding lactose free,200 g,...,200g,3.80,IN_STOCK,True,https://www.fairprice.com.sg/product/arla-prot...,0.666667,0.602740,0.641096,False,LOW
7,redmart_00000036__fairprice_00037226,redmart__fairprice,Snacks,arla,g|1|200.000000|200.000000,redmart,redmart_00000036,Arla Protein Salted Caramel Pudding 200G - Lac...,protein salted caramel pudding lactose free,200 g,...,200g,3.80,IN_STOCK,True,https://www.fairprice.com.sg/product/arla-prot...,0.285714,0.588235,0.406723,False,LOW
8,redmart_00000036__fairprice_00037233,redmart__fairprice,Snacks,arla,g|1|200.000000|200.000000,redmart,redmart_00000036,Arla Protein Salted Caramel Pudding 200G - Lac...,protein salted caramel pudding lactose free,200 g,...,200g,3.80,IN_STOCK,True,https://www.fairprice.com.sg/product/arla-prot...,0.250000,0.547945,0.369178,False,LOW
9,redmart_00000085__fairprice_00037226,redmart__fairprice,Snacks,arla,g|1|200.000000|200.000000,redmart,redmart_00000085,Arla Protein Vanilla Cookie Pudding 200G - Lac...,protein vanilla cookie pudding lactose free,200 g,...,200g,3.80,IN_STOCK,True,https://www.fairprice.com.sg/product/arla-prot...,0.285714,0.588235,0.406723,False,LOW


## 11. Assign Stage 2B-style review recommendations

In [11]:
if len(
    strict_candidates
) > 0:
    strict_candidates[
        "review_recommendation"
    ] = pd.Series(
        np.select(
            [
                strict_candidates[
                    "exact_name_core"
                ].eq(
                    True
                ),

                (
                    strict_candidates[
                        "token_jaccard"
                    ]
                    >= 0.80
                )
                &
                (
                    strict_candidates[
                        "sequence_similarity"
                    ]
                    >= 0.80
                ),

                strict_candidates[
                    "combined_similarity"
                ]
                >= 0.65,
            ],
            [
                "EXACT_NAME_REVIEW",
                "STRONG_VARIANT_REVIEW",
                "POSSIBLE_MATCH_REVIEW",
            ],
            default=(
                "LOW_SIMILARITY_REVIEW"
            ),
        ),
        index=(
            strict_candidates.index
        ),
        dtype="string",
    )

print(
    strict_candidates[
        "review_recommendation"
    ]
    .value_counts()
)


review_recommendation
LOW_SIMILARITY_REVIEW    14575
POSSIBLE_MATCH_REVIEW     2413
EXACT_NAME_REVIEW         1049
STRONG_VARIANT_REVIEW      397
Name: count, dtype: Int64


## 12. Reuse the validated Stage 2C automated identity rules

In [12]:
AUTO_CONFIRM_JACCARD = 0.92
AUTO_CONFIRM_SEQUENCE = 0.90
AUTO_CONFIRM_COMBINED = 0.92

AUTO_REJECT_JACCARD = 0.20
AUTO_REJECT_COMBINED = 0.35

MIN_TOKENS_FOR_LOW_SIM_REJECT = 2


VARIANT_DIMENSIONS = {
    "sugar_style": {
        "ZERO_SUGAR": [
            "zero sugar",
            "sugar free",
            "sugar-free",
            "no sugar",
            "no-sugar",
        ],
        "LESS_SUGAR": [
            "less sugar",
            "reduced sugar",
            "lower sugar",
        ],
        "ORIGINAL_REGULAR": [
            "original",
            "regular",
            "classic",
        ],
    },

    "diet_style": {
        "DIET": [
            "diet",
        ],
        "LIGHT": [
            "light",
            "lite",
        ],
        "REGULAR": [
            "regular",
            "original",
            "classic",
        ],
    },

    "fruit_flavour": {
        "LEMON": [
            "lemon"
        ],
        "LIME": [
            "lime"
        ],
        "ORANGE": [
            "orange"
        ],
        "APPLE": [
            "apple"
        ],
        "GRAPE": [
            "grape"
        ],
        "PEACH": [
            "peach"
        ],
        "LYCHEE": [
            "lychee"
        ],
        "MANGO": [
            "mango"
        ],
        "STRAWBERRY": [
            "strawberry"
        ],
        "BLUEBERRY": [
            "blueberry"
        ],
        "RASPBERRY": [
            "raspberry"
        ],
        "BERRY": [
            "berry"
        ],
        "PINEAPPLE": [
            "pineapple"
        ],
        "WATERMELON": [
            "watermelon"
        ],
    },

    "sweet_flavour": {
        "CHOCOLATE": [
            "chocolate",
            "choco",
        ],
        "VANILLA": [
            "vanilla"
        ],
        "CARAMEL": [
            "caramel"
        ],
        "MATCHA": [
            "matcha"
        ],
        "MOCHA": [
            "mocha"
        ],
        "COFFEE": [
            "coffee"
        ],
    },

    "snack_seasoning": {
        "ORIGINAL": [
            "original",
            "classic",
            "plain",
        ],
        "BBQ": [
            "bbq",
            "barbecue",
        ],
        "CHEESE": [
            "cheese"
        ],
        "SEAWEED": [
            "seaweed"
        ],
        "SOUR_CREAM": [
            "sour cream"
        ],
        "SALTED": [
            "salted"
        ],
        "UNSALTED": [
            "unsalted"
        ],
        "SPICY": [
            "spicy",
            "hot",
            "chilli",
            "chili",
        ],
        "TRUFFLE": [
            "truffle"
        ],
        "GARLIC": [
            "garlic"
        ],
        "HONEY": [
            "honey"
        ],
    },

    "protein_flavour": {
        "CHICKEN": [
            "chicken"
        ],
        "BEEF": [
            "beef"
        ],
        "PORK": [
            "pork"
        ],
        "TUNA": [
            "tuna"
        ],
        "SALMON": [
            "salmon"
        ],
        "SARDINE": [
            "sardine"
        ],
        "MACKEREL": [
            "mackerel"
        ],
    },

    "tea_style": {
        "GREEN_TEA": [
            "green tea"
        ],
        "BLACK_TEA": [
            "black tea"
        ],
        "MILK_TEA": [
            "milk tea"
        ],
        "OOLONG": [
            "oolong"
        ],
        "EARL_GREY": [
            "earl grey"
        ],
    },
}


GENERIC_EXTRA_TOKENS = {
    "drink",
    "drinks",
    "beverage",
    "beverages",
    "snack",
    "snacks",
    "food",
    "foods",
    "can",
    "cans",
    "bottle",
    "bottles",
    "packet",
    "pack",
    "packs",
    "bag",
    "bags",
    "box",
    "boxes",
    "premium",
    "new",
    "imported",
    "value",
    "family",
    "size",
    "originally",
}


def clean_identity_text(
    text,
):
    if pd.isna(
        text
    ):
        return ""

    text = (
        str(
            text
        )
        .casefold()
    )

    text = re.sub(
        r"[^a-z0-9]+",
        " ",
        text,
    )

    text = re.sub(
        r"\s+",
        " ",
        text,
    ).strip()

    return text


def token_set(
    text,
):
    return set(
        clean_identity_text(
            text
        ).split()
    )


def detect_variant_values(
    text,
):
    normalized = (
        clean_identity_text(
            text
        )
    )

    detected = {}

    for (
        dimension,
        values,
    ) in (
        VARIANT_DIMENSIONS.items()
    ):
        found_values = set()

        for (
            value_name,
            phrases,
        ) in values.items():
            for phrase in phrases:
                phrase_normalized = (
                    clean_identity_text(
                        phrase
                    )
                )

                pattern = (
                    r"(?<![a-z0-9])"
                    + re.escape(
                        phrase_normalized
                    )
                    + r"(?![a-z0-9])"
                )

                if re.search(
                    pattern,
                    normalized,
                ):
                    found_values.add(
                        value_name
                    )
                    break

        if found_values:
            detected[
                dimension
            ] = found_values

    return detected


def find_variant_conflicts(
    left_text,
    right_text,
):
    left_variants = (
        detect_variant_values(
            left_text
        )
    )

    right_variants = (
        detect_variant_values(
            right_text
        )
    )

    conflicts = []

    common_dimensions = (
        set(
            left_variants
        )
        &
        set(
            right_variants
        )
    )

    for dimension in sorted(
        common_dimensions
    ):
        left_values = (
            left_variants[
                dimension
            ]
        )

        right_values = (
            right_variants[
                dimension
            ]
        )

        if left_values.isdisjoint(
            right_values
        ):
            conflicts.append(
                {
                    "dimension":
                        dimension,
                    "left_values":
                        sorted(
                            left_values
                        ),
                    "right_values":
                        sorted(
                            right_values
                        ),
                }
            )

    return conflicts


def format_variant_conflicts(
    conflicts,
):
    parts = []

    for conflict in conflicts:
        parts.append(
            (
                f"{conflict['dimension']}: "
                f"{'/'.join(conflict['left_values'])} "
                f"vs "
                f"{'/'.join(conflict['right_values'])}"
            )
        )

    return "; ".join(
        parts
    )


def generic_containment_match(
    left_text,
    right_text,
):
    left_tokens = token_set(
        left_text
    )

    right_tokens = token_set(
        right_text
    )

    if (
        len(
            left_tokens
        ) < 2
        or len(
            right_tokens
        ) < 2
    ):
        return False

    if (
        left_tokens
        <= right_tokens
    ):
        extras = (
            right_tokens
            - left_tokens
        )

    elif (
        right_tokens
        <= left_tokens
    ):
        extras = (
            left_tokens
            - right_tokens
        )

    else:
        return False

    return (
        len(
            extras
        )
        <= 2
        and extras
        <= GENERIC_EXTRA_TOKENS
    )


def automated_identity_decision(
    row,
):
    left_text = (
        clean_identity_text(
            row[
                "left_name_match_text"
            ]
        )
    )

    right_text = (
        clean_identity_text(
            row[
                "right_name_match_text"
            ]
        )
    )

    conflicts = (
        find_variant_conflicts(
            row[
                "left_product_name"
            ],
            row[
                "right_product_name"
            ],
        )
    )

    if conflicts:
        return pd.Series(
            {
                "auto_identity_decision":
                    "AUTO_VARIANT_MISMATCH",
                "auto_confidence":
                    "HIGH",
                "auto_rule_id":
                    "R1_EXPLICIT_VARIANT_CONFLICT",
                "auto_decision_reason":
                    (
                        "Explicit conflicting variant "
                        "terms: "
                        + format_variant_conflicts(
                            conflicts
                        )
                    ),
                "requires_manual_review":
                    False,
            }
        )

    if (
        left_text
        and right_text
        and left_text
        == right_text
    ):
        return pd.Series(
            {
                "auto_identity_decision":
                    "AUTO_MATCH_CONFIRMED",
                "auto_confidence":
                    "HIGH",
                "auto_rule_id":
                    "R2_EXACT_NORMALIZED_CORE",
                "auto_decision_reason":
                    (
                        "Normalized product identity "
                        "text is exactly equal."
                    ),
                "requires_manual_review":
                    False,
            }
        )

    left_tokens = (
        token_set(
            left_text
        )
    )

    right_tokens = (
        token_set(
            right_text
        )
    )

    if (
        left_tokens
        and right_tokens
        and left_tokens
        == right_tokens
    ):
        return pd.Series(
            {
                "auto_identity_decision":
                    "AUTO_MATCH_CONFIRMED",
                "auto_confidence":
                    "HIGH",
                "auto_rule_id":
                    "R3_IDENTICAL_TOKEN_SET",
                "auto_decision_reason":
                    (
                        "Product identity token sets "
                        "are identical."
                    ),
                "requires_manual_review":
                    False,
            }
        )

    jaccard = float(
        row[
            "token_jaccard"
        ]
    )

    sequence = float(
        row[
            "sequence_similarity"
        ]
    )

    combined = float(
        row[
            "combined_similarity"
        ]
    )

    if (
        jaccard
        >= AUTO_CONFIRM_JACCARD
        and sequence
        >= AUTO_CONFIRM_SEQUENCE
        and combined
        >= AUTO_CONFIRM_COMBINED
    ):
        return pd.Series(
            {
                "auto_identity_decision":
                    "AUTO_MATCH_CONFIRMED",
                "auto_confidence":
                    "HIGH",
                "auto_rule_id":
                    "R4_VERY_HIGH_NAME_SIMILARITY",
                "auto_decision_reason":
                    (
                        "Same brand and strict pack, "
                        "no detected variant conflict, "
                        "and very high lexical agreement."
                    ),
                "requires_manual_review":
                    False,
            }
        )

    if (
        generic_containment_match(
            left_text,
            right_text,
        )
        and sequence
        >= 0.80
        and combined
        >= 0.80
    ):
        return pd.Series(
            {
                "auto_identity_decision":
                    "AUTO_MATCH_CONFIRMED",
                "auto_confidence":
                    "HIGH",
                "auto_rule_id":
                    "R5_GENERIC_WORDING_CONTAINMENT",
                "auto_decision_reason":
                    (
                        "One identity name is contained "
                        "in the other and extra tokens "
                        "are generic descriptors."
                    ),
                "requires_manual_review":
                    False,
            }
        )

    if (
        len(
            left_tokens
        )
        >= MIN_TOKENS_FOR_LOW_SIM_REJECT
        and len(
            right_tokens
        )
        >= MIN_TOKENS_FOR_LOW_SIM_REJECT
        and jaccard
        <= AUTO_REJECT_JACCARD
        and combined
        <= AUTO_REJECT_COMBINED
    ):
        return pd.Series(
            {
                "auto_identity_decision":
                    "AUTO_NOT_SAME_PRODUCT",
                "auto_confidence":
                    "HIGH",
                "auto_rule_id":
                    "R6_VERY_LOW_NAME_SIMILARITY",
                "auto_decision_reason":
                    (
                        "Same brand and pack block, "
                        "but product names have very "
                        "low identity similarity."
                    ),
                "requires_manual_review":
                    False,
            }
        )

    return pd.Series(
        {
            "auto_identity_decision":
                "NEEDS_MANUAL_REVIEW",
            "auto_confidence":
                "REVIEW",
            "auto_rule_id":
                "R7_BORDERLINE_OR_INSUFFICIENT_EVIDENCE",
            "auto_decision_reason":
                (
                    "Automated evidence is not strong "
                    "enough for a high-confidence decision."
                ),
            "requires_manual_review":
                True,
        }
    )


## 13. Run all-category automated identity assessment

In [13]:
automation_result = (
    strict_candidates
    .apply(
        automated_identity_decision,
        axis=1,
    )
)

all_category_pairs = pd.concat(
    [
        strict_candidates.copy(),
        automation_result,
    ],
    axis=1,
)

print(
    "All-category automated identity decisions:"
)

print(
    all_category_pairs[
        "auto_identity_decision"
    ]
    .value_counts()
)

display(
    all_category_pairs[
        [
            "target_category",
            "left_product_name",
            "right_product_name",
            "combined_similarity",
            "auto_identity_decision",
            "auto_rule_id",
        ]
    ].head(30)
)


All-category automated identity decisions:
auto_identity_decision
NEEDS_MANUAL_REVIEW      12613
AUTO_NOT_SAME_PRODUCT     2959
AUTO_MATCH_CONFIRMED      1534
AUTO_VARIANT_MISMATCH     1328
Name: count, dtype: int64


,target_category,left_product_name,right_product_name,combined_similarity,auto_identity_decision,auto_rule_id
0,Snacks,Arla Protein Chocolate Pudding 200G - Lactose ...,Arla Protein Pudding - Chocolate,0.575873,NEEDS_MANUAL_REVIEW,R7_BORDERLINE_OR_INSUFFICIENT_EVIDENCE
1,Snacks,Arla Protein Chocolate Pudding 200G - Lactose ...,Arla Protein Pudding - Hazelnut Latte,0.406723,NEEDS_MANUAL_REVIEW,R7_BORDERLINE_OR_INSUFFICIENT_EVIDENCE
2,Snacks,Arla Protein Chocolate Pudding 200G - Lactose ...,Arla Protein Pudding - Salted Caramel,0.394958,AUTO_VARIANT_MISMATCH,R1_EXPLICIT_VARIANT_CONFLICT
3,Snacks,Arla Protein Hazelnut Pudding 200G - Lactose Free,Arla Protein Pudding - Hazelnut Latte,0.550746,NEEDS_MANUAL_REVIEW,R7_BORDERLINE_OR_INSUFFICIENT_EVIDENCE
4,Snacks,Arla Protein Hazelnut Pudding 200G - Lactose Free,Arla Protein Pudding - Chocolate,0.458065,NEEDS_MANUAL_REVIEW,R7_BORDERLINE_OR_INSUFFICIENT_EVIDENCE
5,Snacks,Arla Protein Hazelnut Pudding 200G - Lactose Free,Arla Protein Pudding - Salted Caramel,0.398294,NEEDS_MANUAL_REVIEW,R7_BORDERLINE_OR_INSUFFICIENT_EVIDENCE
6,Snacks,Arla Protein Salted Caramel Pudding 200G - Lac...,Arla Protein Pudding - Salted Caramel,0.641096,NEEDS_MANUAL_REVIEW,R7_BORDERLINE_OR_INSUFFICIENT_EVIDENCE
7,Snacks,Arla Protein Salted Caramel Pudding 200G - Lac...,Arla Protein Pudding - Chocolate,0.406723,AUTO_VARIANT_MISMATCH,R1_EXPLICIT_VARIANT_CONFLICT
8,Snacks,Arla Protein Salted Caramel Pudding 200G - Lac...,Arla Protein Pudding - Hazelnut Latte,0.369178,NEEDS_MANUAL_REVIEW,R7_BORDERLINE_OR_INSUFFICIENT_EVIDENCE
9,Snacks,Arla Protein Vanilla Cookie Pudding 200G - Lac...,Arla Protein Pudding - Chocolate,0.406723,AUTO_VARIANT_MISMATCH,R1_EXPLICIT_VARIANT_CONFLICT


## 14. Apply Stage 2C reviewed pilot decisions as authoritative overrides

Candidate-pair IDs are deterministic because the retailer-pair orientation is fixed.

If a candidate pair was already reviewed/resolved in Stage 2C, its final Stage 2C decision overrides the new automatic decision.

Unreviewed Stage 2C pairs do not override Stage 2D automation.


In [14]:
VALID_STAGE2C_OVERRIDE_DECISIONS = {
    "MATCH_CONFIRMED",
    "VARIANT_MISMATCH",
    "NOT_SAME_PRODUCT",
    "EXCLUDE_OTHER",
    "NEEDS_MORE_EVIDENCE",
}

stage2c_override = (
    stage2c_final_pairs.loc[
        stage2c_final_pairs[
            "final_identity_decision"
        ].isin(
            VALID_STAGE2C_OVERRIDE_DECISIONS
        ),
        [
            "candidate_pair_id",
            "final_identity_decision",
            "final_decision_source",
        ]
    ]
    .drop_duplicates(
        subset=[
            "candidate_pair_id"
        ]
    )
    .rename(
        columns={
            "final_identity_decision":
                "stage2c_final_identity_decision",
            "final_decision_source":
                "stage2c_final_decision_source",
        }
    )
)

expanded_final_pairs = (
    all_category_pairs
    .merge(
        stage2c_override,
        on="candidate_pair_id",
        how="left",
    )
)

AUTO_TO_FINAL = {
    "AUTO_MATCH_CONFIRMED":
        "MATCH_CONFIRMED",
    "AUTO_VARIANT_MISMATCH":
        "VARIANT_MISMATCH",
    "AUTO_NOT_SAME_PRODUCT":
        "NOT_SAME_PRODUCT",
    "NEEDS_MANUAL_REVIEW":
        "UNREVIEWED",
}

expanded_final_pairs[
    "final_identity_decision"
] = (
    expanded_final_pairs[
        "auto_identity_decision"
    ]
    .map(
        AUTO_TO_FINAL
    )
    .astype("string")
)

expanded_final_pairs[
    "final_decision_source"
] = (
    "STAGE2D_AUTOMATED"
)

pilot_override_mask = (
    expanded_final_pairs[
        "stage2c_final_identity_decision"
    ]
    .notna()
)

expanded_final_pairs.loc[
    pilot_override_mask,
    "final_identity_decision"
] = expanded_final_pairs.loc[
    pilot_override_mask,
    "stage2c_final_identity_decision"
]

expanded_final_pairs.loc[
    pilot_override_mask,
    "final_decision_source"
] = "STAGE2C_REVIEWED_OVERRIDE"

print(
    "Stage 2C reviewed overrides applied:",
    f"{int(pilot_override_mask.sum()):,}"
)

print(
    "\nFinal all-category identity decisions:"
)

print(
    expanded_final_pairs[
        "final_identity_decision"
    ]
    .value_counts(
        dropna=False
    )
)


Stage 2C reviewed overrides applied: 3,624

Final all-category identity decisions:
final_identity_decision
UNREVIEWED          12613
NOT_SAME_PRODUCT     2929
MATCH_CONFIRMED      1536
VARIANT_MISMATCH     1356
Name: count, dtype: Int64


## 15. Quarantine ambiguous and rejected candidate pairs

In [15]:
manual_identity_review = (
    expanded_final_pairs.loc[
        expanded_final_pairs[
            "final_identity_decision"
        ]
        .isin(
            [
                "UNREVIEWED",
                "NEEDS_MORE_EVIDENCE",
            ]
        )
    ]
    .copy()
)

rejected_identity_pairs = (
    expanded_final_pairs.loc[
        expanded_final_pairs[
            "final_identity_decision"
        ]
        .isin(
            [
                "VARIANT_MISMATCH",
                "NOT_SAME_PRODUCT",
                "EXCLUDE_OTHER",
            ]
        )
    ]
    .copy()
)

identity_confirmed_pairs = (
    expanded_final_pairs.loc[
        expanded_final_pairs[
            "final_identity_decision"
        ]
        == "MATCH_CONFIRMED"
    ]
    .copy()
)

print(
    "Identity-confirmed pairs:",
    f"{len(identity_confirmed_pairs):,}"
)

print(
    "Ambiguous/manual-review pairs:",
    f"{len(manual_identity_review):,}"
)

print(
    "Rejected/non-match pairs:",
    f"{len(rejected_identity_pairs):,}"
)


Identity-confirmed pairs: 1,536
Ambiguous/manual-review pairs: 12,613
Rejected/non-match pairs: 4,285


## 16. Generate all-category pack-discrepancy quarantine

In [16]:
RELAXED_BLOCK_COLUMNS = [
    "target_category_final",
    "brand_normalized",
    "total_pack_key",
]

MAX_RELAXED_PAIRS_PER_BLOCK = 2_000
MAX_RELAXED_CANDIDATES_PER_SOURCE = 3

pack_discrepancy_records = []

relaxed_grouped = (
    expansion.groupby(
        RELAXED_BLOCK_COLUMNS,
        dropna=False,
        sort=False,
    )
)

for (
    block_key,
    block,
) in relaxed_grouped:
    retailers_here = set(
        block[
            "retailer"
        ]
        .unique()
    )

    if len(
        retailers_here
    ) < 2:
        continue

    if (
        block[
            "strict_pack_key"
        ]
        .nunique()
        <= 1
    ):
        continue

    for (
        left_retailer,
        right_retailer,
    ) in RETAILER_PAIRS:
        if (
            left_retailer
            not in retailers_here
            or right_retailer
            not in retailers_here
        ):
            continue

        left_df = (
            block.loc[
                block[
                    "retailer"
                ]
                == left_retailer
            ]
        )

        right_df = (
            block.loc[
                block[
                    "retailer"
                ]
                == right_retailer
            ]
        )

        large_block = (
            len(
                left_df
            )
            * len(
                right_df
            )
            > MAX_RELAXED_PAIRS_PER_BLOCK
        )

        for (
            _,
            left_row,
        ) in left_df.iterrows():
            local_candidates = []

            for (
                _,
                right_row,
            ) in right_df.iterrows():
                if (
                    left_row[
                        "strict_pack_key"
                    ]
                    == right_row[
                        "strict_pack_key"
                    ]
                ):
                    continue

                if (
                    large_block
                    and not shared_meaningful_token(
                        left_row[
                            "name_match_text"
                        ],
                        right_row[
                            "name_match_text"
                        ],
                    )
                ):
                    continue

                (
                    jaccard,
                    sequence,
                    combined,
                    exact_core,
                ) = score_name_pair(
                    left_row[
                        "name_match_text"
                    ],
                    right_row[
                        "name_match_text"
                    ],
                )

                if combined < 0.50:
                    continue

                local_candidates.append(
                    {
                        "retailer_pair":
                            (
                                f"{left_retailer}"
                                f"__{right_retailer}"
                            ),
                        "target_category":
                            left_row[
                                "target_category_final"
                            ],
                        "brand_normalized":
                            left_row[
                                "brand_normalized"
                            ],
                        "left_source_record_id":
                            str(
                                left_row[
                                    "source_record_id"
                                ]
                            ),
                        "left_product_name":
                            left_row[
                                "product_name_raw"
                            ],
                        "left_specification":
                            left_row[
                                "specification_raw"
                            ],
                        "left_strict_pack_key":
                            left_row[
                                "strict_pack_key"
                            ],
                        "right_source_record_id":
                            str(
                                right_row[
                                    "source_record_id"
                                ]
                            ),
                        "right_product_name":
                            right_row[
                                "product_name_raw"
                            ],
                        "right_specification":
                            right_row[
                                "specification_raw"
                            ],
                        "right_strict_pack_key":
                            right_row[
                                "strict_pack_key"
                            ],
                        "total_pack_key":
                            left_row[
                                "total_pack_key"
                            ],
                        "combined_similarity":
                            round(
                                float(
                                    combined
                                ),
                                6,
                            ),
                        "review_reason":
                            (
                                "SAME_BRAND_AND_TOTAL_QUANTITY_"
                                "BUT_PACK_STRUCTURE_DIFFERS"
                            ),
                    }
                )

            local_candidates = sorted(
                local_candidates,
                key=lambda row: row[
                    "combined_similarity"
                ],
                reverse=True,
            )[
                :MAX_RELAXED_CANDIDATES_PER_SOURCE
            ]

            pack_discrepancy_records.extend(
                local_candidates
            )


pack_discrepancy_review = pd.DataFrame(
    pack_discrepancy_records
)

if len(
    pack_discrepancy_review
) > 0:
    pack_discrepancy_review = (
        pack_discrepancy_review
        .drop_duplicates(
            subset=[
                "left_source_record_id",
                "right_source_record_id",
            ]
        )
        .reset_index(
            drop=True
        )
    )

print(
    "All-category pack-discrepancy quarantine:",
    f"{len(pack_discrepancy_review):,}"
)


All-category pack-discrepancy quarantine: 307


## 17. Build provisional all-category canonical identity groups

In [17]:
class UnionFind:
    def __init__(
        self,
    ):
        self.parent = {}
        self.rank = {}

    def add(
        self,
        item,
    ):
        if (
            item
            not in self.parent
        ):
            self.parent[
                item
            ] = item

            self.rank[
                item
            ] = 0

    def find(
        self,
        item,
    ):
        self.add(
            item
        )

        if (
            self.parent[
                item
            ]
            != item
        ):
            self.parent[
                item
            ] = self.find(
                self.parent[
                    item
                ]
            )

        return self.parent[
            item
        ]

    def union(
        self,
        left,
        right,
    ):
        root_left = (
            self.find(
                left
            )
        )

        root_right = (
            self.find(
                right
            )
        )

        if (
            root_left
            == root_right
        ):
            return

        left_rank = (
            self.rank[
                root_left
            ]
        )

        right_rank = (
            self.rank[
                root_right
            ]
        )

        if (
            left_rank
            < right_rank
        ):
            self.parent[
                root_left
            ] = root_right

        elif (
            left_rank
            > right_rank
        ):
            self.parent[
                root_right
            ] = root_left

        else:
            self.parent[
                root_right
            ] = root_left

            self.rank[
                root_left
            ] += 1


uf = UnionFind()

for row in (
    identity_confirmed_pairs
    .itertuples(
        index=False
    )
):
    uf.union(
        str(
            row.left_source_record_id
        ),
        str(
            row.right_source_record_id
        ),
    )


component_members = {}

for source_id in uf.parent:
    root = uf.find(
        source_id
    )

    component_members.setdefault(
        root,
        [],
    ).append(
        source_id
    )

print(
    "Provisional all-category canonical components:",
    f"{len(component_members):,}"
)


Provisional all-category canonical components: 1,414


## 18. Create canonical membership table

In [18]:
expansion_lookup = (
    expansion.copy()
)

expansion_lookup[
    "source_record_id"
] = (
    expansion_lookup[
        "source_record_id"
    ]
    .astype(str)
)

expansion_lookup = (
    expansion_lookup
    .set_index(
        "source_record_id",
        drop=False,
    )
)

membership_rows = []

for (
    root,
    members,
) in component_members.items():
    members = sorted(
        set(
            str(
                member
            )
            for member in members
        )
    )

    stable_text = (
        "||".join(
            members
        )
    )

    canonical_hash = (
        hashlib.sha256(
            stable_text.encode(
                "utf-8"
            )
        )
        .hexdigest()[
            :12
        ]
    )

    canonical_product_id = (
        f"CP_{canonical_hash}"
    )

    for source_id in members:
        if (
            source_id
            not in expansion_lookup.index
        ):
            continue

        row = (
            expansion_lookup.loc[
                source_id
            ]
        )

        membership_rows.append(
            {
                "canonical_product_id":
                    canonical_product_id,
                "source_record_id":
                    source_id,
                "retailer":
                    row[
                        "retailer"
                    ],
                "target_category":
                    row[
                        "target_category_final"
                    ],
                "brand_normalized":
                    row[
                        "brand_normalized"
                    ],
                "product_name_raw":
                    row[
                        "product_name_raw"
                    ],
                "specification_raw":
                    row[
                        "specification_raw"
                    ],
                "strict_pack_key":
                    row[
                        "strict_pack_key"
                    ],
                "product_price":
                    row[
                        "product_price"
                    ],
                "stock_status":
                    row[
                        "stock_status"
                    ],
                "currently_available":
                    row[
                        "currently_available"
                    ],
                "offer_eligible":
                    row[
                        "offer_eligible"
                    ],
                "product_url_raw":
                    row[
                        "product_url_raw"
                    ],
            }
        )


canonical_membership = pd.DataFrame(
    membership_rows
)

print(
    "Canonical membership rows:",
    f"{len(canonical_membership):,}"
)


Canonical membership rows: 2,908


## 19. Validate canonical group consistency

In [19]:
canonical_group_rows = []

if len(
    canonical_membership
) > 0:
    for (
        canonical_product_id,
        group,
    ) in canonical_membership.groupby(
        "canonical_product_id"
    ):
        category_count = (
            group[
                "target_category"
            ]
            .nunique(
                dropna=False
            )
        )

        brand_count = (
            group[
                "brand_normalized"
            ]
            .nunique(
                dropna=False
            )
        )

        pack_key_count = (
            group[
                "strict_pack_key"
            ]
            .nunique(
                dropna=False
            )
        )

        retailer_count = (
            group[
                "retailer"
            ]
            .nunique()
        )

        duplicate_retailer_count = int(
            (
                group[
                    "retailer"
                ]
                .value_counts()
                > 1
            ).sum()
        )

        preliminary_active_offer_retailer_count = (
            group.loc[
                group[
                    "offer_eligible"
                ]
                .fillna(False),
                "retailer"
            ]
            .nunique()
        )

        if (
            category_count
            != 1
            or brand_count
            != 1
            or pack_key_count
            != 1
        ):
            group_status = (
                "IDENTITY_CONFLICT"
            )

        elif (
            duplicate_retailer_count
            > 0
        ):
            group_status = (
                "REVIEW_DUPLICATE_RETAILER"
            )

        else:
            group_status = (
                "IDENTITY_VERIFIED"
            )

        canonical_group_rows.append(
            {
                "canonical_product_id":
                    canonical_product_id,
                "member_count":
                    len(
                        group
                    ),
                "retailer_count":
                    retailer_count,
                "category_count":
                    category_count,
                "brand_count":
                    brand_count,
                "pack_key_count":
                    pack_key_count,
                "duplicate_retailer_count":
                    duplicate_retailer_count,
                "preliminary_active_offer_retailer_count":
                    preliminary_active_offer_retailer_count,
                "group_status":
                    group_status,
            }
        )


canonical_group_checks = pd.DataFrame(
    canonical_group_rows
)

print(
    "Canonical group status:"
)

print(
    canonical_group_checks[
        "group_status"
    ]
    .value_counts()
)


Canonical group status:
group_status
IDENTITY_VERIFIED            1388
REVIEW_DUPLICATE_RETAILER      26
Name: count, dtype: int64


## 20. Build all-category provisional canonical product table

In [20]:
canonical_product_rows = []

for check in (
    canonical_group_checks
    .itertuples(
        index=False
    )
):
    group = (
        canonical_membership.loc[
            canonical_membership[
                "canonical_product_id"
            ]
            == check.canonical_product_id
        ]
        .copy()
    )

    if len(
        group
    ) == 0:
        continue

    group = (
        group.sort_values(
            [
                "retailer",
                "source_record_id",
            ]
        )
    )

    first = (
        group.iloc[
            0
        ]
    )

    canonical_product_rows.append(
        {
            "canonical_product_id":
                check.canonical_product_id,

            # Representative source wording only;
            # not asserted as a final canonical display name.
            "representative_product_name":
                first[
                    "product_name_raw"
                ],

            "canonical_name":
                pd.NA,

            "target_category":
                first[
                    "target_category"
                ],
            "brand_normalized":
                first[
                    "brand_normalized"
                ],
            "strict_pack_key":
                first[
                    "strict_pack_key"
                ],
            "member_count":
                check.member_count,
            "retailer_count":
                check.retailer_count,
            "preliminary_active_offer_retailer_count":
                check.preliminary_active_offer_retailer_count,
            "group_status":
                check.group_status,
            "prelim_2plus_active_offer_retailers":
                (
                    check.preliminary_active_offer_retailer_count
                    >= 2
                ),
        }
    )


canonical_products = pd.DataFrame(
    canonical_product_rows
)

print(
    "All-category provisional canonical products:",
    f"{len(canonical_products):,}"
)

display(
    pd.crosstab(
        canonical_products[
            "target_category"
        ],
        canonical_products[
            "group_status"
        ],
        margins=True,
    )
)


All-category provisional canonical products: 1,414


group_status,IDENTITY_VERIFIED,REVIEW_DUPLICATE_RETAILER,All
target_category,,,
Breakfast Foods,55,0,55
Canned & Packaged Food,183,0,183
Dairy,112,2,114
Drinks,168,3,171
Household Cleaning,110,1,111
Rice & Noodles,204,2,206
Snacks,465,14,479
Toiletries & Personal Care,91,4,95
All,1388,26,1414


## 21. Select a  provisional 1300-product shortlist

Only `IDENTITY_VERIFIED` groups can enter the shortlist automatically.

Ranking prioritises:

1. 2+ preliminary active-offer retailers;
2. 3-retailer identity coverage;
3. more preliminary active-offer retailers;
4. deterministic canonical-product ID tie-breaking.

The shortlist first tries to place at least 50 products in every category. Remaining places are filled globally, with a maximum of 450 per category.


In [21]:
TARGET_SHORTLIST_SIZE = 1300
MIN_PRODUCTS_PER_CATEGORY = 50
MAX_PRODUCTS_PER_CATEGORY = 450

eligible_products = (
    canonical_products.loc[
        canonical_products[
            "group_status"
        ]
        == "IDENTITY_VERIFIED"
    ]
    .copy()
)

eligible_products[
    "selection_quality_tier"
] = np.select(
    [
        (
            eligible_products[
                "preliminary_active_offer_retailer_count"
            ]
            >= 2
        )
        &
        (
            eligible_products[
                "retailer_count"
            ]
            == 3
        ),

        (
            eligible_products[
                "preliminary_active_offer_retailer_count"
            ]
            >= 2
        ),

        (
            eligible_products[
                "preliminary_active_offer_retailer_count"
            ]
            == 1
        ),
    ],
    [
        4,
        3,
        2,
    ],
    default=1,
)


def stable_shortlist_hash(
    canonical_product_id,
):
    digest = hashlib.sha256(
        str(
            canonical_product_id
        ).encode(
            "utf-8"
        )
    ).hexdigest()

    return int(
        digest[
            :16
        ],
        16,
    )


eligible_products[
    "_stable_selection_hash"
] = (
    eligible_products[
        "canonical_product_id"
    ]
    .map(
        stable_shortlist_hash
    )
)

eligible_products = (
    eligible_products
    .sort_values(
        [
            "selection_quality_tier",
            "retailer_count",
            "preliminary_active_offer_retailer_count",
            "_stable_selection_hash",
        ],
        ascending=[
            False,
            False,
            False,
            True,
        ]
    )
    .reset_index(
        drop=True
    )
)

selected_ids = set()
selected_counts = {
    category: 0
    for category
    in TARGET_CATEGORIES
}

# ---------------------------------------------------------
# Pass 1: minimum representation per category.
# ---------------------------------------------------------

for category in TARGET_CATEGORIES:
    category_pool = (
        eligible_products.loc[
            eligible_products[
                "target_category"
            ]
            == category
        ]
    )

    take_n = min(
        MIN_PRODUCTS_PER_CATEGORY,
        len(
            category_pool
        ),
    )

    chosen = (
        category_pool.head(
            take_n
        )
    )

    for canonical_id in (
        chosen[
            "canonical_product_id"
        ]
    ):
        selected_ids.add(
            canonical_id
        )

    selected_counts[
        category
    ] += len(
        chosen
    )


# ---------------------------------------------------------
# Pass 2: fill remaining places while respecting max/category.
# ---------------------------------------------------------

for row in (
    eligible_products
    .itertuples(
        index=False
    )
):
    if (
        len(
            selected_ids
        )
        >= TARGET_SHORTLIST_SIZE
    ):
        break

    canonical_id = (
        row.canonical_product_id
    )

    category = (
        row.target_category
    )

    if (
        canonical_id
        in selected_ids
    ):
        continue

    if (
        selected_counts[
            category
        ]
        >= MAX_PRODUCTS_PER_CATEGORY
    ):
        continue

    selected_ids.add(
        canonical_id
    )

    selected_counts[
        category
    ] += 1


shortlist = (
    eligible_products.loc[
        eligible_products[
            "canonical_product_id"
        ]
        .isin(
            selected_ids
        )
    ]
    .copy()
)

shortlist[
    "shortlist_status"
] = (
    "AWAITING_FULFILMENT_VERIFICATION"
)

shortlist[
    "shortlist_rank_within_category"
] = (
    shortlist
    .groupby(
        "target_category"
    )
    .cumcount()
    + 1
)

shortlist = (
    shortlist
    .sort_values(
        [
            "target_category",
            "shortlist_rank_within_category",
        ]
    )
    .reset_index(
        drop=True
    )
)

print(
    "Provisional shortlist size:",
    f"{len(shortlist):,}"
)

print(
    "Target shortlist size:",
    TARGET_SHORTLIST_SIZE
)

display(
    shortlist[
        "target_category"
    ]
    .value_counts()
    .reindex(
        TARGET_CATEGORIES,
        fill_value=0,
    )
    .rename(
        "selected_products"
    )
    .to_frame()
)


Provisional shortlist size: 1,300
Target shortlist size: 1300


,selected_products
target_category,
Rice & Noodles,194
Drinks,154
Dairy,103
Breakfast Foods,52
Canned & Packaged Food,174
Snacks,431
Household Cleaning,104
Toiletries & Personal Care,88


## 22. Build shortlist category coverage and shortfall report

In [22]:
available_by_category = (
    eligible_products[
        "target_category"
    ]
    .value_counts()
    .reindex(
        TARGET_CATEGORIES,
        fill_value=0,
    )
)

selected_by_category = (
    shortlist[
        "target_category"
    ]
    .value_counts()
    .reindex(
        TARGET_CATEGORIES,
        fill_value=0,
    )
)

selected_prelim_2plus = (
    shortlist.loc[
        shortlist[
            "prelim_2plus_active_offer_retailers"
        ]
    ]
    [
        "target_category"
    ]
    .value_counts()
    .reindex(
        TARGET_CATEGORIES,
        fill_value=0,
    )
)

shortlist_category_coverage = pd.DataFrame(
    {
        "target_category":
            TARGET_CATEGORIES,
        "available_identity_verified_products":
            [
                int(
                    available_by_category[
                        category
                    ]
                )
                for category
                in TARGET_CATEGORIES
            ],
        "selected_products":
            [
                int(
                    selected_by_category[
                        category
                    ]
                )
                for category
                in TARGET_CATEGORIES
            ],
        "minimum_target":
            MIN_PRODUCTS_PER_CATEGORY,
        "minimum_shortfall":
            [
                max(
                    0,
                    MIN_PRODUCTS_PER_CATEGORY
                    - int(
                        selected_by_category[
                            category
                        ]
                    ),
                )
                for category
                in TARGET_CATEGORIES
            ],
        "selected_prelim_2plus_active_offer_retailers":
            [
                int(
                    selected_prelim_2plus[
                        category
                    ]
                )
                for category
                in TARGET_CATEGORIES
            ],
    }
)

shortlist_category_coverage[
    "selected_prelim_2plus_rate"
] = np.where(
    shortlist_category_coverage[
        "selected_products"
    ]
    > 0,
    (
        shortlist_category_coverage[
            "selected_prelim_2plus_active_offer_retailers"
        ]
        /
        shortlist_category_coverage[
            "selected_products"
        ]
    ),
    np.nan,
)

display(
    shortlist_category_coverage
)

overall_prelim_2plus_count = int(
    shortlist[
        "prelim_2plus_active_offer_retailers"
    ].sum()
)

overall_prelim_2plus_rate = (
    overall_prelim_2plus_count
    / len(
        shortlist
    )
    if len(
        shortlist
    ) > 0
    else np.nan
)

print(
    "Shortlist products with preliminary "
    "2+ active-offer retailers:",
    f"{overall_prelim_2plus_count:,}"
)

print(
    "Preliminary 2+ retailer rate:",
    (
        f"{overall_prelim_2plus_rate:.2%}"
        if not pd.isna(
            overall_prelim_2plus_rate
        )
        else "N/A"
    )
)

print(
    "\nImportant: this is NOT the final fulfilment-verified "
    "2+ retailer coverage metric."
)


,target_category,available_identity_verified_products,selected_products,minimum_target,minimum_shortfall,selected_prelim_2plus_active_offer_retailers,selected_prelim_2plus_rate
0,Rice & Noodles,204,194,50,0,191,0.984536
1,Drinks,168,154,50,0,149,0.967532
2,Dairy,112,103,50,0,101,0.980583
3,Breakfast Foods,55,52,50,0,51,0.980769
4,Canned & Packaged Food,183,174,50,0,168,0.965517
5,Snacks,465,431,50,0,413,0.958237
6,Household Cleaning,110,104,50,0,98,0.942308
7,Toiletries & Personal Care,91,88,50,0,87,0.988636


Shortlist products with preliminary 2+ active-offer retailers: 1,258
Preliminary 2+ retailer rate: 96.77%

Important: this is NOT the final fulfilment-verified 2+ retailer coverage metric.


## 23. Build focused fulfilment-verification queue for shortlist listings

In [23]:
shortlist_membership = (
    canonical_membership.loc[
        canonical_membership[
            "canonical_product_id"
        ]
        .isin(
            shortlist[
                "canonical_product_id"
            ]
        )
    ]
    .copy()
)


def url_host(
    url,
):
    if pd.isna(
        url
    ):
        return ""

    try:
        return (
            urlparse(
                str(
                    url
                )
            )
            .netloc
            .casefold()
            .replace(
                "www.",
                "",
            )
        )
    except Exception:
        return ""


shortlist_membership[
    "source_url_host"
] = (
    shortlist_membership[
        "product_url_raw"
    ]
    .map(
        url_host
    )
)


def fulfilment_precheck(
    row,
):
    retailer = (
        str(
            row[
                "retailer"
            ]
        )
        .casefold()
    )

    host = (
        str(
            row[
                "source_url_host"
            ]
        )
        .casefold()
    )

    if (
        retailer
        == "fairprice"
        and (
            host
            == "fairprice.com.sg"
            or host.endswith(
                ".fairprice.com.sg"
            )
        )
    ):
        return (
            "OFFICIAL_RETAILER_DOMAIN_"
            "DIRECT_FULFILMENT_STILL_TO_VERIFY"
        )

    if (
        retailer
        == "coldstorage"
        and (
            host
            == "coldstorage.com.sg"
            or host.endswith(
                ".coldstorage.com.sg"
            )
        )
    ):
        return (
            "OFFICIAL_RETAILER_DOMAIN_"
            "DIRECT_FULFILMENT_STILL_TO_VERIFY"
        )

    if (
        retailer
        == "redmart"
        and (
            host
            == "lazada.sg"
            or host.endswith(
                ".lazada.sg"
            )
        )
    ):
        return (
            "MARKETPLACE_DOMAIN_"
            "SELLER_FULFILMENT_MUST_BE_VERIFIED"
        )

    return (
        "MANUAL_FULFILMENT_VERIFICATION_REQUIRED"
    )


shortlist_membership[
    "fulfilment_precheck"
] = (
    shortlist_membership
    .apply(
        fulfilment_precheck,
        axis=1,
    )
)

STAGE2D_FULFILMENT_REVIEW_FILE = (
    CURATED_DATA_DIR
    / "stage2d_shortlist_fulfilment_review_working.csv"
)

FULFILMENT_MANUAL_COLUMNS = [
    "fulfilment_status",
    "fulfilment_evidence",
    "fulfilment_reviewer",
    "fulfilment_notes",
]

shortlist_membership[
    "fulfilment_status"
] = "UNREVIEWED"

for column in (
    FULFILMENT_MANUAL_COLUMNS[
        1:
    ]
):
    shortlist_membership[
        column
    ] = pd.NA


# ---------------------------------------------------------
# Carry forward any Stage 2C fulfilment work where available.
# ---------------------------------------------------------

if (
    STAGE2C_FULFILMENT_FILE.exists()
):
    stage2c_fulfilment = pd.read_csv(
        STAGE2C_FULFILMENT_FILE,
        dtype="string",
        low_memory=False,
    )

    stage2c_fulfilment_values = (
        stage2c_fulfilment[
            [
                "source_record_id",
                *FULFILMENT_MANUAL_COLUMNS,
            ]
        ]
        .drop_duplicates(
            subset=[
                "source_record_id"
            ]
        )
    )

    shortlist_membership = (
        shortlist_membership
        .drop(
            columns=FULFILMENT_MANUAL_COLUMNS
        )
        .merge(
            stage2c_fulfilment_values,
            on="source_record_id",
            how="left",
        )
    )

    shortlist_membership[
        "fulfilment_status"
    ] = (
        shortlist_membership[
            "fulfilment_status"
        ]
        .fillna(
            "UNREVIEWED"
        )
    )


# ---------------------------------------------------------
# Preserve Stage 2D fulfilment decisions across reruns.
# Stage 2D entries take precedence over carried Stage 2C values.
# ---------------------------------------------------------

if (
    STAGE2D_FULFILMENT_REVIEW_FILE.exists()
):
    existing_stage2d_fulfilment = (
        pd.read_csv(
            STAGE2D_FULFILMENT_REVIEW_FILE,
            dtype="string",
            low_memory=False,
        )
    )

    existing_stage2d_values = (
        existing_stage2d_fulfilment[
            [
                "source_record_id",
                *FULFILMENT_MANUAL_COLUMNS,
            ]
        ]
        .drop_duplicates(
            subset=[
                "source_record_id"
            ]
        )
    )

    shortlist_membership = (
        shortlist_membership
        .merge(
            existing_stage2d_values,
            on="source_record_id",
            how="left",
            suffixes=(
                "",
                "_stage2d_existing",
            ),
        )
    )

    for column in (
        FULFILMENT_MANUAL_COLUMNS
    ):
        existing_column = (
            f"{column}"
            f"_stage2d_existing"
        )

        if (
            existing_column
            in shortlist_membership.columns
        ):
            existing_has_value = (
                shortlist_membership[
                    existing_column
                ]
                .notna()
                &
                shortlist_membership[
                    existing_column
                ]
                .astype("string")
                .str.strip()
                .ne("")
            )

            shortlist_membership.loc[
                existing_has_value,
                column,
            ] = shortlist_membership.loc[
                existing_has_value,
                existing_column,
            ]

            shortlist_membership = (
                shortlist_membership.drop(
                    columns=[
                        existing_column
                    ]
                )
            )


shortlist_membership.to_csv(
    STAGE2D_FULFILMENT_REVIEW_FILE,
    index=False,
    encoding="utf-8",
)

print(
    "Shortlist fulfilment-verification listing rows:",
    f"{len(shortlist_membership):,}"
)

print(
    "\nFulfilment status:"
)

print(
    shortlist_membership[
        "fulfilment_status"
    ]
    .value_counts(
        dropna=False
    )
)


Shortlist fulfilment-verification listing rows: 2,650

Fulfilment status:
fulfilment_status
UNREVIEWED    2650
Name: count, dtype: Int64


## 24. Calculate current fulfilment-verified shortlist coverage

In [24]:
VALID_FULFILMENT_STATUSES = {
    "UNREVIEWED",
    "VERIFIED_DIRECT",
    "NOT_DIRECT",
    "UNKNOWN",
}

invalid_fulfilment_statuses = sorted(
    set(
        shortlist_membership[
            "fulfilment_status"
        ]
        .dropna()
        .astype(str)
        .unique()
    )
    - VALID_FULFILMENT_STATUSES
)

assert not invalid_fulfilment_statuses, (
    "Invalid fulfilment status value(s): "
    f"{invalid_fulfilment_statuses}"
)

shortlist_membership[
    "verified_direct_listing"
] = (
    shortlist_membership[
        "fulfilment_status"
    ]
    == "VERIFIED_DIRECT"
)

offer_eligible_boolean = (
    shortlist_membership[
        "offer_eligible"
    ]
    .astype("string")
    .str.strip()
    .str.casefold()
    .map(
        {
            "true": True,
            "false": False,
            "yes": True,
            "no": False,
            "1": True,
            "0": False,
        }
    )
    .fillna(False)
)

shortlist_membership[
    "verified_active_offer"
] = (
    shortlist_membership[
        "verified_direct_listing"
    ]
    &
    offer_eligible_boolean
)

verified_shortlist_rows = []

for (
    canonical_product_id,
    group,
) in shortlist_membership.groupby(
    "canonical_product_id"
):
    verified_direct_retailers = (
        group.loc[
            group[
                "verified_direct_listing"
            ],
            "retailer"
        ]
        .nunique()
    )

    verified_active_offer_retailers = (
        group.loc[
            group[
                "verified_active_offer"
            ],
            "retailer"
        ]
        .nunique()
    )

    verified_shortlist_rows.append(
        {
            "canonical_product_id":
                canonical_product_id,
            "verified_direct_retailer_count":
                verified_direct_retailers,
            "verified_active_offer_retailer_count":
                verified_active_offer_retailers,
            "meets_verified_2plus_offer_target":
                (
                    verified_active_offer_retailers
                    >= 2
                ),
        }
    )


verified_shortlist_coverage = pd.DataFrame(
    verified_shortlist_rows
)

shortlist_final_status = (
    shortlist
    .merge(
        verified_shortlist_coverage,
        on="canonical_product_id",
        how="left",
    )
)

for column in [
    "verified_direct_retailer_count",
    "verified_active_offer_retailer_count",
]:
    shortlist_final_status[
        column
    ] = (
        shortlist_final_status[
            column
        ]
        .fillna(0)
        .astype(int)
    )

shortlist_final_status[
    "meets_verified_2plus_offer_target"
] = (
    shortlist_final_status[
        "meets_verified_2plus_offer_target"
    ]
    .fillna(False)
)

verified_2plus_count = int(
    shortlist_final_status[
        "meets_verified_2plus_offer_target"
    ].sum()
)

verified_2plus_rate = (
    verified_2plus_count
    / len(
        shortlist_final_status
    )
    if len(
        shortlist_final_status
    ) > 0
    else np.nan
)

print(
    "Shortlist products with 2+ verified active offers:",
    f"{verified_2plus_count:,}"
)

print(
    "Current verified 2+ retailer rate:",
    (
        f"{verified_2plus_rate:.2%}"
        if not pd.isna(
            verified_2plus_rate
        )
        else "N/A"
    )
)

print(
    "\nThis value is expected to be low/zero until "
    "the shortlist fulfilment review is completed."
)


Shortlist products with 2+ verified active offers: 0
Current verified 2+ retailer rate: 0.00%

This value is expected to be low/zero until the shortlist fulfilment review is completed.


## 25. Identify expansion records with no strict cross-retailer candidate

In [25]:
candidate_source_ids = set()

if len(
    strict_candidates
) > 0:
    candidate_source_ids.update(
        strict_candidates[
            "left_source_record_id"
        ]
        .astype(str)
    )

    candidate_source_ids.update(
        strict_candidates[
            "right_source_record_id"
        ]
        .astype(str)
    )

expansion[
    "has_strict_cross_retailer_candidate"
] = (
    expansion[
        "source_record_id"
    ]
    .astype(str)
    .isin(
        candidate_source_ids
    )
)

unmatched_catalog_records = (
    expansion.loc[
        ~expansion[
            "has_strict_cross_retailer_candidate"
        ]
    ]
    .copy()
)

print(
    "All-category catalog records without a "
    "strict cross-retailer candidate:",
    f"{len(unmatched_catalog_records):,}"
)


All-category catalog records without a strict cross-retailer candidate: 38,051


## 26. Save Stage 2D outputs

In [26]:
EXPANSION_POOL_FILE = (
    CURATED_DATA_DIR
    / "stage2d_all_category_expansion_pool.csv"
)

STRICT_CANDIDATES_FILE = (
    CURATED_DATA_DIR
    / "stage2d_all_category_strict_candidates.csv"
)

FINAL_PAIRS_FILE = (
    CURATED_DATA_DIR
    / "stage2d_all_category_final_identity_pairs.csv"
)

IDENTITY_CONFIRMED_FILE = (
    CURATED_DATA_DIR
    / "stage2d_identity_confirmed_pairs.csv"
)

MANUAL_REVIEW_FILE = (
    CURATED_DATA_DIR
    / "stage2d_manual_identity_review.csv"
)

REJECTED_PAIRS_FILE = (
    CURATED_DATA_DIR
    / "stage2d_rejected_identity_pairs.csv"
)

PACK_DISCREPANCY_FILE = (
    CURATED_DATA_DIR
    / "stage2d_pack_discrepancy_review.csv"
)

CANONICAL_MEMBERSHIP_FILE = (
    CURATED_DATA_DIR
    / "stage2d_canonical_membership_provisional.csv"
)

CANONICAL_GROUP_CHECKS_FILE = (
    CURATED_DATA_DIR
    / "stage2d_canonical_group_checks.csv"
)

CANONICAL_PRODUCTS_FILE = (
    CURATED_DATA_DIR
    / "stage2d_canonical_products_provisional.csv"
)

SHORTLIST_FILE = (
    CURATED_DATA_DIR
    / "stage2d_provisional_seed_shortlist.csv"
)

SHORTLIST_FINAL_STATUS_FILE = (
    CURATED_DATA_DIR
    / "stage2d_provisional_seed_shortlist_status.csv"
)

SHORTLIST_CATEGORY_COVERAGE_FILE = (
    CURATED_DATA_DIR
    / "stage2d_shortlist_category_coverage.csv"
)

UNMATCHED_FILE = (
    CURATED_DATA_DIR
    / "stage2d_unmatched_catalog_records.csv"
)

BLOCK_SUMMARY_FILE = (
    CURATED_DATA_DIR
    / "stage2d_strict_block_summary.csv"
)

expansion.to_csv(
    EXPANSION_POOL_FILE,
    index=False,
    encoding="utf-8",
)

strict_candidates.to_csv(
    STRICT_CANDIDATES_FILE,
    index=False,
    encoding="utf-8",
)

expanded_final_pairs.to_csv(
    FINAL_PAIRS_FILE,
    index=False,
    encoding="utf-8",
)

identity_confirmed_pairs.to_csv(
    IDENTITY_CONFIRMED_FILE,
    index=False,
    encoding="utf-8",
)

manual_identity_review.to_csv(
    MANUAL_REVIEW_FILE,
    index=False,
    encoding="utf-8",
)

rejected_identity_pairs.to_csv(
    REJECTED_PAIRS_FILE,
    index=False,
    encoding="utf-8",
)

pack_discrepancy_review.to_csv(
    PACK_DISCREPANCY_FILE,
    index=False,
    encoding="utf-8",
)

canonical_membership.to_csv(
    CANONICAL_MEMBERSHIP_FILE,
    index=False,
    encoding="utf-8",
)

canonical_group_checks.to_csv(
    CANONICAL_GROUP_CHECKS_FILE,
    index=False,
    encoding="utf-8",
)

canonical_products.to_csv(
    CANONICAL_PRODUCTS_FILE,
    index=False,
    encoding="utf-8",
)

shortlist.to_csv(
    SHORTLIST_FILE,
    index=False,
    encoding="utf-8",
)

shortlist_final_status.to_csv(
    SHORTLIST_FINAL_STATUS_FILE,
    index=False,
    encoding="utf-8",
)

shortlist_category_coverage.to_csv(
    SHORTLIST_CATEGORY_COVERAGE_FILE,
    index=False,
    encoding="utf-8",
)

unmatched_catalog_records.to_csv(
    UNMATCHED_FILE,
    index=False,
    encoding="utf-8",
)

strict_block_summary.to_csv(
    BLOCK_SUMMARY_FILE,
    index=False,
    encoding="utf-8",
)

print(
    "Saved Stage 2D outputs."
)


Saved Stage 2D outputs.


## 27. Verify Stage 2D output files

In [27]:
OUTPUT_FILES = [
    EXPANSION_POOL_FILE,
    STRICT_CANDIDATES_FILE,
    FINAL_PAIRS_FILE,
    IDENTITY_CONFIRMED_FILE,
    MANUAL_REVIEW_FILE,
    REJECTED_PAIRS_FILE,
    PACK_DISCREPANCY_FILE,
    CANONICAL_MEMBERSHIP_FILE,
    CANONICAL_GROUP_CHECKS_FILE,
    CANONICAL_PRODUCTS_FILE,
    SHORTLIST_FILE,
    SHORTLIST_FINAL_STATUS_FILE,
    SHORTLIST_CATEGORY_COVERAGE_FILE,
    STAGE2D_FULFILMENT_REVIEW_FILE,
    UNMATCHED_FILE,
    BLOCK_SUMMARY_FILE,
]

for file_path in OUTPUT_FILES:
    print(
        f"{file_path.name:58s} -> "
        f"{file_path.exists()}"
    )
    assert file_path.exists()

print(
    "\nPASS: All Stage 2D output files were created."
)


stage2d_all_category_expansion_pool.csv                    -> True
stage2d_all_category_strict_candidates.csv                 -> True
stage2d_all_category_final_identity_pairs.csv              -> True
stage2d_identity_confirmed_pairs.csv                       -> True
stage2d_manual_identity_review.csv                         -> True
stage2d_rejected_identity_pairs.csv                        -> True
stage2d_pack_discrepancy_review.csv                        -> True
stage2d_canonical_membership_provisional.csv               -> True
stage2d_canonical_group_checks.csv                         -> True
stage2d_canonical_products_provisional.csv                 -> True
stage2d_provisional_seed_shortlist.csv                     -> True
stage2d_provisional_seed_shortlist_status.csv              -> True
stage2d_shortlist_category_coverage.csv                    -> True
stage2d_shortlist_fulfilment_review_working.csv            -> True
stage2d_unmatched_catalog_records.csv                      -> 

## 28. Final Stage 2D validation

In [28]:
verified_group_count = int(
    (
        canonical_group_checks[
            "group_status"
        ]
        == "IDENTITY_VERIFIED"
    ).sum()
)

duplicate_retailer_group_count = int(
    (
        canonical_group_checks[
            "group_status"
        ]
        == "REVIEW_DUPLICATE_RETAILER"
    ).sum()
)

identity_conflict_group_count = int(
    (
        canonical_group_checks[
            "group_status"
        ]
        == "IDENTITY_CONFLICT"
    ).sum()
)

selected_categories_count = (
    shortlist[
        "target_category"
    ]
    .nunique()
)

minimum_category_shortfalls = int(
    (
        shortlist_category_coverage[
            "minimum_shortfall"
        ]
        > 0
    ).sum()
)

print("=" * 100)
print("STAGE 2D FINAL VALIDATION — ALL EIGHT CATEGORY EXPANSION")
print("=" * 100)

print(
    f"Stage 2C QA family agreement rate             : "
    f"{qa_family_agreement_rate:.2%}"
)

print(
    f"Stage 2C unresolved material conflicts        : "
    f"{qa_unresolved_conflicts:,}"
)

print("-" * 100)

print(
    f"All-category catalog expansion records        : "
    f"{len(expansion):,}"
)

print(
    f"Target categories represented                 : "
    f"{expansion['target_category_final'].nunique():,}/8"
)

print(
    f"Strict candidate pairs                        : "
    f"{len(strict_candidates):,}"
)

print(
    f"Stage 2C reviewed overrides applied           : "
    f"{int(pilot_override_mask.sum()):,}"
)

print(
    f"Final MATCH_CONFIRMED pairs                   : "
    f"{len(identity_confirmed_pairs):,}"
)

print(
    f"Ambiguous/manual-review pairs                 : "
    f"{len(manual_identity_review):,}"
)

print(
    f"Pack-discrepancy quarantine pairs             : "
    f"{len(pack_discrepancy_review):,}"
)

print("-" * 100)

print(
    f"Provisional canonical products                : "
    f"{len(canonical_products):,}"
)

print(
    f"Identity-verified canonical products          : "
    f"{verified_group_count:,}"
)

print(
    f"Duplicate-retailer groups requiring review    : "
    f"{duplicate_retailer_group_count:,}"
)

print(
    f"Identity-conflict groups                       : "
    f"{identity_conflict_group_count:,}"
)

print("-" * 100)

print(
    f"Provisional seed shortlist                    : "
    f"{len(shortlist):,}"
)

print(
    f"Shortlist categories represented              : "
    f"{selected_categories_count:,}/8"
)

print(
    f"Categories below minimum target               : "
    f"{minimum_category_shortfalls:,}"
)

print(
    f"Shortlist preliminary 2+ active-offer products: "
    f"{overall_prelim_2plus_count:,}"
)

print(
    f"Preliminary 2+ retailer rate                  : "
    f"{overall_prelim_2plus_rate:.2%}"
)

print(
    f"Fulfilment-verification listing rows          : "
    f"{len(shortlist_membership):,}"
)

print(
    f"Current verified 2+ active-offer products     : "
    f"{verified_2plus_count:,}"
)

print(
    f"Current verified 2+ active-offer rate         : "
    f"{verified_2plus_rate:.2%}"
)

print("=" * 100)

assert (
    qa_family_agreement_rate
    >= MIN_QA_FAMILY_AGREEMENT_RATE
)

assert (
    qa_unresolved_conflicts
    == 0
)

assert (
    expansion[
        "target_category_final"
    ]
    .nunique()
    == 8
)

assert (
    set(
        expansion[
            "target_category_final"
        ]
        .unique()
    )
    == set(
        TARGET_CATEGORIES
    )
)

assert (
    strict_candidates[
        "candidate_pair_id"
    ]
    .duplicated()
    .sum()
    == 0
)

assert (
    (
        identity_confirmed_pairs[
            "final_identity_decision"
        ]
        == "MATCH_CONFIRMED"
    )
    .all()
)

assert (
    len(
        shortlist
    )
    <= TARGET_SHORTLIST_SIZE
)

assert (
    selected_categories_count
    == 8
), (
    "The provisional seed shortlist must represent "
    "all eight target categories."
)

print()
print(
    "PASS: Stage 2D all-category expansion and "
    "balanced provisional shortlist succeeded."
)

print()

if (
    len(
        shortlist
    )
    < TARGET_SHORTLIST_SIZE
):
    print(
        "NOTE: Fewer than 1300 identity-verified products "
        "were available under the current conservative rules. "
        "See the category shortfall report before deciding "
        "whether to review ambiguous pairs."
    )

if (
    minimum_category_shortfalls
    > 0
):
    print(
        "ACTION: One or more categories have fewer than "
        "20 selected products. Review the shortfall table "
        "and consider targeted manual review only for those "
        "categories."
    )

print(
    "NEXT: Verify direct supermarket fulfilment only for "
    "the focused shortlist listing queue in "
    "stage2d_shortlist_fulfilment_review_working.csv."
)

print(
    "After fulfilment verification, rerun Stage 2D and use "
    "the verified 2+ active-offer coverage to decide whether "
    "targeted replacement/manual review is needed before "
    "final seed publication."
)


STAGE 2D FINAL VALIDATION — ALL EIGHT CATEGORY EXPANSION
Stage 2C QA family agreement rate             : 97.78%
Stage 2C unresolved material conflicts        : 0
----------------------------------------------------------------------------------------------------
All-category catalog expansion records        : 49,690
Target categories represented                 : 8/8
Strict candidate pairs                        : 18,434
Stage 2C reviewed overrides applied           : 3,624
Final MATCH_CONFIRMED pairs                   : 1,536
Ambiguous/manual-review pairs                 : 12,613
Pack-discrepancy quarantine pairs             : 307
----------------------------------------------------------------------------------------------------
Provisional canonical products                : 1,414
Identity-verified canonical products          : 1,388
Duplicate-retailer groups requiring review    : 26
Identity-conflict groups                       : 0
-------------------------------------------------

## Stage 2D result and next action

After the first Stage 2D run, inspect:

```text
stage2d_shortlist_category_coverage.csv
stage2d_provisional_seed_shortlist.csv
stage2d_shortlist_fulfilment_review_working.csv
```

### What not to do

Do not manually review every ambiguous all-category pair.

The shortlist is deliberately built from high-confidence identity-verified groups first.

Only perform targeted manual identity review when:

- a category has a shortlist shortfall; or
- fulfilment verification causes too many selected products to lose 2+ retailer coverage.

### Fulfilment review

For the shortlist fulfilment queue, use:

```text
VERIFIED_DIRECT
NOT_DIRECT
UNKNOWN
UNREVIEWED
```

Record reviewer/evidence for verified decisions.

After the fulfilment queue is reviewed, rerun Stage 2D. The notebook will then report the actual percentage of shortlisted canonical products with verified active offers from at least two retailers.

That verified figure, not the preliminary `offer_eligible` figure, should be used for the final seed acceptance check.
